# 00 DQA Status and Full Supervised Baseline

このノートブックは、現在のDQA系実験の完了状況をまとめて確認し、`04` / `04_2` の8時間設定と同じデータ規模・学習量で、DQAやphase分割を使わない通常の教師あり学習を実行するためのものです。

既定では `04 DQA-CWA v2` と同じ `15 warmup + 14 phase1 + 27 phase2 = 56` を、単一の fully supervised training の `epochs=56` として使います。参考値として、DQAの画像提示回数にだけ合わせる場合の約45 epochも下で計算します。

## 1. Paths and Runtime Defaults

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from collections import Counter, deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_v2.py",
        "navigating_data_heterogeneity/setup_fedsto_exact_reproduction.py",
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend(
            [
                base,
                base / "Object_Detection",
                base / "masters_research" / "Object_Detection",
            ]
        )
    for candidate in candidate_dirs:
        if all((candidate / marker).exists() for marker in required):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Object_Detection repository root.")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
NAV_ROOT = REPO_ROOT / "navigating_data_heterogeneity"
ET_ROOT = NAV_ROOT / "vendor" / "efficientteacher"
EVAL_SCRIPT = DQA_ROOT / "evaluate_paper_protocol.py"

WORK_ROOT = DQA_ROOT / "efficientteacher_dqa_full_supervised_00"
DATA_ROOT = WORK_ROOT / "full_supervised_data"
LIST_ROOT = WORK_ROOT / "data_lists"
CONFIG_ROOT = WORK_ROOT / "configs"
RUN_ROOT = WORK_ROOT / "runs"
GLOBAL_DIR = WORK_ROOT / "global_checkpoints"
RUN_NAME = "dqa_full_supervised_04_8h_56epochs"
CONFIG_PATH = CONFIG_ROOT / "runtime_dqa_full_supervised_04_8h.yaml"
FULL_TRAIN_LIST = LIST_ROOT / "full_supervised_train.txt"
FULL_MANIFEST_PATH = WORK_ROOT / "full_supervised_manifest.json"
FINAL_CHECKPOINT = GLOBAL_DIR / "full_supervised_04_8h_56epochs.pt"

RUNNER_LOG = DQA_ROOT / "dqa_full_supervised_00_runner.out"
TRAIN_LOG = DQA_ROOT / "dqa_full_supervised_00_train.log"
PID_PATH = DQA_ROOT / "dqa_full_supervised_00_runner.pid"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("dqa_root:", DQA_ROOT)
print("workspace:", WORK_ROOT)
print("python:", PYTHON_BIN)

## 2. Current DQA Status

`03` / `04` / `04_3` 系の履歴、stats、DQA guard、評価出力の有無をまとめます。ここは読み取り専用です。

In [ ]:
def load_json(path: Path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return {"_error": str(exc)}


def modified_utc(path: Path) -> str:
    if not path.exists():
        return ""
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat()


def checkpoint_exists(path_text: str | None) -> bool:
    return bool(path_text) and Path(path_text).exists() and Path(path_text).stat().st_size >= 1024 * 1024


DQA_WORKSPACES = [
    {
        "label": "03 corrected 8h",
        "workspace": DQA_ROOT / "efficientteacher_dqa_cwa_corrected_12h",
        "stats": DQA_ROOT / "stats_corrected_12h",
        "expected_phase1": 14,
        "expected_phase2": 27,
    },
    {
        "label": "04 v2 8h",
        "workspace": DQA_ROOT / "efficientteacher_dqa_ver2",
        "stats": DQA_ROOT / "stats_dqa_ver2",
        "expected_phase1": 14,
        "expected_phase2": 27,
    },
    {
        "label": "04_3 v2 scene 12h",
        "workspace": DQA_ROOT / "efficientteacher_dqa_ver2_scene_12h",
        "stats": DQA_ROOT / "stats_dqa_ver2_scene_12h",
        "expected_phase1": 20,
        "expected_phase2": 35,
    },
]

rows = []
for item in DQA_WORKSPACES:
    workspace = item["workspace"]
    stats_root = item["stats"]
    manifest = load_json(workspace / "manifest.json", {})
    history = load_json(workspace / "history.json", [])
    state = load_json(workspace / "dqa_cwa_state.json", {})
    guard_history = (state.get("round_guard") or {}).get("history") or [] if isinstance(state, dict) else []
    latest_global = history[-1].get("global") if isinstance(history, list) and history else None
    phase1 = sum(1 for entry in history if isinstance(entry, dict) and int(entry.get("phase", 0)) == 1)
    phase2 = sum(1 for entry in history if isinstance(entry, dict) and int(entry.get("phase", 0)) == 2)
    clients = manifest.get("clients", []) if isinstance(manifest, dict) else []
    eval_summary = workspace / "validation_reports" / "paper_protocol_eval_summary.csv"
    rows.append(
        {
            "run": item["label"],
            "workspace_exists": workspace.exists(),
            "server_train": (manifest.get("server") or {}).get("train_images") if isinstance(manifest, dict) else None,
            "server_val": (manifest.get("server") or {}).get("val_images") if isinstance(manifest, dict) else None,
            "client_images": sum(int(client.get("images", 0)) for client in clients),
            "clients": ", ".join(str(client.get("scene") or client.get("weather")) for client in clients),
            "phase1": f"{phase1}/{item['expected_phase1']}",
            "phase2": f"{phase2}/{item['expected_phase2']}",
            "total_rounds": f"{len(history) if isinstance(history, list) else 0}/{item['expected_phase1'] + item['expected_phase2']}",
            "latest_checkpoint_ok": checkpoint_exists(latest_global),
            "round_stats": len(list(stats_root.glob("phase*_round[0-9][0-9][0-9].json"))),
            "client_stats": len(list(stats_root.glob("phase*_round*_client*.json"))),
            "dqa_used": sum(1 for entry in guard_history if entry.get("used_dqa")),
            "dqa_skipped": sum(1 for entry in guard_history if not entry.get("used_dqa")),
            "paper_eval_csv": eval_summary.exists(),
            "modified_utc": modified_utc(workspace / "history.json"),
        }
    )

status_df = pd.DataFrame(rows)
display(status_df)

In [ ]:
pseudo_rows = []
for item in DQA_WORKSPACES:
    stats_root = item["stats"]
    for stats_file in sorted(stats_root.glob("phase*_round[0-9][0-9][0-9].json")):
        payload = load_json(stats_file, {})
        clients = payload.get("clients", []) if isinstance(payload, dict) else []
        counts_by_class = [0.0] * 10
        weighted_quality = 0.0
        total_count = 0.0
        for client in clients:
            counts = [max(float(value), 0.0) for value in client.get("counts", [])[:10]]
            qualities = [float(value) for value in client.get("mean_quality_scores", [])[:10]]
            for idx, count in enumerate(counts):
                counts_by_class[idx] += count
                total_count += count
                if idx < len(qualities):
                    weighted_quality += count * min(max(qualities[idx], 0.0), 1.0)
        match = re.search(r"phase(\d+)_round(\d+)", stats_file.name)
        pseudo_rows.append(
            {
                "run": item["label"],
                "phase": int(match.group(1)) if match else None,
                "round": int(match.group(2)) if match else None,
                "pseudo_boxes": total_count,
                "active_classes": sum(1 for value in counts_by_class if value > 0),
                "mean_quality": weighted_quality / total_count if total_count > 0 else 0.0,
            }
        )

if pseudo_rows:
    pseudo_df = pd.DataFrame(pseudo_rows)
    summary = (
        pseudo_df.groupby("run")
        .agg(
            rounds=("round", "count"),
            pseudo_boxes_total=("pseudo_boxes", "sum"),
            pseudo_boxes_mean=("pseudo_boxes", "mean"),
            pseudo_boxes_min=("pseudo_boxes", "min"),
            pseudo_boxes_max=("pseudo_boxes", "max"),
            mean_quality_mean=("mean_quality", "mean"),
            active_classes_min=("active_classes", "min"),
            active_classes_max=("active_classes", "max"),
        )
        .reset_index()
    )
    display(summary)
    display(pseudo_df.tail(10))
else:
    print("No DQA pseudo-label stats files were found.")

## 3. 04/04_2-Matched Supervised Schedule

`04` の8時間DQA v2設定を基準にします。phaseは作らず、全データを1本の教師ありtrain listにまとめ、`epochs=56` で回します。

In [ ]:
WARMUP_EPOCHS = 15
PHASE1_ROUNDS = 14
PHASE2_ROUNDS = 27
TOTAL_SUPERVISED_EPOCHS = WARMUP_EPOCHS + PHASE1_ROUNDS + PHASE2_ROUNDS
BATCH_SIZE = 64
WORKERS = 0
REQUESTED_GPUS = 2
MIN_FREE_GIB = 70
MASTER_PORT = find_free_port(29516)

RUN_FULL_SUPERVISED = True
ALLOW_CPU_TRAINING = False
FORCE_REBUILD_DATA = False
FORCE_RETRAIN = False
APPEND_TRAIN_LOG = False
STATUS_EVERY_SECONDS = 60

try:
    gpu_probe = subprocess.run(
        [str(PYTHON_BIN), "-c", "import torch; print(torch.cuda.device_count())"],
        capture_output=True,
        text=True,
        check=True,
    )
    AVAILABLE_CUDA_GPUS = int(gpu_probe.stdout.strip() or "0")
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices with PYTHON_BIN:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(
        f"Requested {REQUESTED_GPUS} GPUs, but {AVAILABLE_CUDA_GPUS} CUDA device(s) are visible. "
        f"Using GPUS={GPUS} to avoid DDP launch failure."
    )

dqa_server_train = 4881
dqa_client_train = 15000
dqa_full_train = dqa_server_train + dqa_client_train
dqa_image_presentations = WARMUP_EPOCHS * dqa_server_train + (PHASE1_ROUNDS + PHASE2_ROUNDS) * dqa_full_train
sample_equivalent_epochs = math.ceil(dqa_image_presentations / dqa_full_train)

{
    "warmup_epochs_reference": WARMUP_EPOCHS,
    "phase1_rounds_reference": PHASE1_ROUNDS,
    "phase2_rounds_reference": PHASE2_ROUNDS,
    "supervised_epochs_default": TOTAL_SUPERVISED_EPOCHS,
    "sample_equivalent_epochs_reference": sample_equivalent_epochs,
    "dqa_image_presentations_reference": dqa_image_presentations,
    "full_supervised_train_images_reference": dqa_full_train,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "requested_gpus": REQUESTED_GPUS,
    "available_cuda_gpus": AVAILABLE_CUDA_GPUS,
    "gpus": GPUS,
    "master_port": MASTER_PORT,
    "min_free_gib": MIN_FREE_GIB,
}

## 4. Build Full-Supervised Data Interface

DQAではclient側15,000枚はunlabeled扱いですが、ここではBDD100Kのraw annotationからYOLOラベルを作り、server 4,881枚 + client 15,000枚を `full_supervised_train.txt` にまとめます。画像はworkspace内にsymlinkし、元データは変更しません。

In [ ]:
if str(NAV_ROOT) not in sys.path:
    sys.path.insert(0, str(NAV_ROOT))

import setup_fedsto_exact_reproduction as setup
import prepare_bdd100k_for_fedsto as prep
import yaml
from PIL import Image

setup.WORK_ROOT = WORK_ROOT
setup.LIST_ROOT = LIST_ROOT
setup.CONFIG_ROOT = CONFIG_ROOT
setup.RUN_ROOT = RUN_ROOT

manifest = setup.build_data_lists()
manifest_summary = {
    "server_train_images": manifest["server"]["train_images"],
    "server_val_images": manifest["server"]["val_images"],
    "clients": [
        {"id": client["id"], "weather": client["weather"], "images": client["images"]}
        for client in manifest["clients"]
    ],
    "paper_eval_total_images": manifest["paper_evaluation"]["total"]["images"],
}
manifest_summary

In [ ]:
def image_to_label_path(image_path: Path) -> Path:
    parts = list(image_path.parts)
    try:
        image_index = parts.index("images")
    except ValueError:
        text = str(image_path)
        if "images_unlabeled" in text:
            return Path(text.replace("images_unlabeled", "labels_unlabeled", 1)).with_suffix(".txt")
        raise ValueError(f"Image path does not contain an images directory: {image_path}") from None
    parts[image_index] = "labels"
    return Path(*parts).with_suffix(".txt")


def link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src)
    except OSError:
        shutil.copy2(src, dst)


def read_list(path: Path) -> list[Path]:
    return [Path(line.strip()) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_label(path: Path, lines: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(("\n".join(lines) + "\n") if lines else "", encoding="utf-8")


def remove_cache_for_list(path: Path) -> None:
    for cache_path in (path.with_suffix(".cache"), path.parent / f"{path.stem}.cache"):
        if cache_path.exists():
            cache_path.unlink()


def build_full_supervised_dataset(force: bool = False) -> dict:
    expected_images = int(manifest["server"]["train_images"]) + sum(int(client["images"]) for client in manifest["clients"])
    if FULL_MANIFEST_PATH.exists() and FULL_TRAIN_LIST.exists() and not force:
        existing = load_json(FULL_MANIFEST_PATH, {})
        listed = len(read_list(FULL_TRAIN_LIST))
        if existing.get("total_images") == expected_images and listed == expected_images:
            print(f"Reusing existing full-supervised dataset: {FULL_TRAIN_LIST}")
            return existing

    if DATA_ROOT.exists() and force:
        shutil.rmtree(DATA_ROOT)
    (DATA_ROOT / "images" / "train").mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / "labels" / "train").mkdir(parents=True, exist_ok=True)

    raw_root = prep.dataset_root()
    train_records = {record["name"]: record for record in prep.load_records(raw_root, "train")}

    image_rows = []
    class_counts = Counter()
    empty_labels = 0
    missing_records = 0
    total_boxes = 0

    server_images = read_list(LIST_ROOT / "server_cloudy_train.txt")
    for src in server_images:
        dst_name = f"server__{src.name}"
        dst_img = DATA_ROOT / "images" / "train" / dst_name
        dst_label = DATA_ROOT / "labels" / "train" / f"{Path(dst_name).stem}.txt"
        label_src = image_to_label_path(src)
        if not label_src.exists():
            raise FileNotFoundError(f"Missing server label for full supervised baseline: {label_src}")
        lines = [line for line in label_src.read_text(encoding="utf-8").splitlines() if line.strip()]
        link_or_copy(src, dst_img)
        write_label(dst_label, lines)
        for line in lines:
            class_counts[setup.BDD_NAMES[int(line.split()[0])]] += 1
        total_boxes += len(lines)
        if not lines:
            empty_labels += 1
        image_rows.append(str(dst_img.resolve()))

    client_summaries = []
    for client in manifest["clients"]:
        client_id = int(client["id"])
        client_list = Path(client["list"])
        client_images = read_list(client_list)
        client_boxes = 0
        client_empty = 0
        for src in client_images:
            dst_name = f"client{client_id}__{src.name}"
            dst_img = DATA_ROOT / "images" / "train" / dst_name
            dst_label = DATA_ROOT / "labels" / "train" / f"{Path(dst_name).stem}.txt"
            record = train_records.get(src.name)
            if record is None:
                lines = []
                missing_records += 1
            else:
                with Image.open(src) as img:
                    lines = prep.mapped_yolo_lines(record, img.size)
            link_or_copy(src, dst_img)
            write_label(dst_label, lines)
            for line in lines:
                class_counts[setup.BDD_NAMES[int(line.split()[0])]] += 1
            total_boxes += len(lines)
            client_boxes += len(lines)
            if not lines:
                empty_labels += 1
                client_empty += 1
            image_rows.append(str(dst_img.resolve()))
        client_summaries.append(
            {
                "id": client_id,
                "weather": client["weather"],
                "images": len(client_images),
                "boxes": client_boxes,
                "empty_label_images": client_empty,
            }
        )

    FULL_TRAIN_LIST.parent.mkdir(parents=True, exist_ok=True)
    FULL_TRAIN_LIST.write_text("\n".join(image_rows) + "\n", encoding="utf-8")
    remove_cache_for_list(FULL_TRAIN_LIST)

    full_manifest = {
        "protocol": "dqa_full_supervised_04_8h_v1",
        "source_dqa_reference": "04_dqa_ver2_reproduction.ipynb",
        "train_list": str(FULL_TRAIN_LIST.resolve()),
        "data_root": str(DATA_ROOT.resolve()),
        "server_images": len(server_images),
        "client_images": sum(item["images"] for item in client_summaries),
        "total_images": len(image_rows),
        "total_boxes": total_boxes,
        "empty_label_images": empty_labels,
        "missing_raw_records": missing_records,
        "class_counts": dict(class_counts),
        "clients": client_summaries,
        "epochs_default": TOTAL_SUPERVISED_EPOCHS,
        "sample_equivalent_epochs_reference": sample_equivalent_epochs,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    FULL_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    FULL_MANIFEST_PATH.write_text(json.dumps(full_manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return full_manifest


full_manifest = build_full_supervised_dataset(force=FORCE_REBUILD_DATA)
display(pd.DataFrame([full_manifest]).drop(columns=["class_counts", "clients"], errors="ignore"))
display(pd.DataFrame(full_manifest["clients"]))

## 5. Write the Supervised Training Config

モデル、optimizer、augmentation、batch size、image size、validation listはDQA/FedSTO側の `efficientteacher_config` をそのまま使い、`target=None` にしてSSOD/DQAを無効化します。

In [ ]:
PRETRAINED_URL = "https://github.com/AlibabaResearch/efficientteacher/releases/download/1.0/efficient-yolov5l.pt"
PRETRAINED_PATH = WORK_ROOT / "weights" / "efficient-yolov5l.pt"


def ensure_pretrained() -> Path:
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    if PRETRAINED_PATH.exists() and PRETRAINED_PATH.stat().st_size >= 1024 * 1024:
        return PRETRAINED_PATH
    candidates = [
        DQA_ROOT / "efficientteacher_dqa_ver2" / "weights" / "efficient-yolov5l.pt",
        DQA_ROOT / "efficientteacher_dqa_cwa_corrected_12h" / "weights" / "efficient-yolov5l.pt",
        NAV_ROOT / "efficientteacher_fedsto" / "weights" / "efficient-yolov5l.pt",
    ]
    for candidate in candidates:
        if candidate.exists() and candidate.stat().st_size >= 1024 * 1024:
            print("Reusing pretrained weights from:", candidate)
            link_or_copy(candidate.resolve(), PRETRAINED_PATH)
            return PRETRAINED_PATH
    print(f"Downloading {PRETRAINED_URL} -> {PRETRAINED_PATH}")
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)
    return PRETRAINED_PATH


required = {
    "yaml": "PyYAML",
    "cv2": "opencv-python",
    "thop": "thop",
    "tensorboard": "tensorboard",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "torch": "torch",
}
dependency_probe = """
import importlib.util, json, sys
required = __REQUIRED__
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
print(json.dumps(missing))
sys.exit(1 if missing else 0)
""".replace("__REQUIRED__", repr(required))
dependency_result = subprocess.run(
    [str(PYTHON_BIN), "-c", dependency_probe],
    capture_output=True,
    text=True,
)
missing = []
try:
    missing = json.loads(dependency_result.stdout.strip() or "[]")
except json.JSONDecodeError:
    pass
if dependency_result.returncode != 0 or missing:
    detail = ", ".join(missing) if missing else dependency_result.stderr.strip()
    raise ModuleNotFoundError(f"Missing runtime dependencies in {PYTHON_BIN}: {detail}")

pretrained = ensure_pretrained()
config_device = "" if GPUS > 1 else ""
cfg = setup.efficientteacher_config(
    name=RUN_NAME,
    train=FULL_TRAIN_LIST,
    val=LIST_ROOT / "server_cloudy_val.txt",
    target=None,
    weights=str(pretrained.resolve()),
    epochs=TOTAL_SUPERVISED_EPOCHS,
    train_scope="all",
    orthogonal_weight=0.0,
    batch_size=BATCH_SIZE,
    workers=WORKERS,
    device=config_device,
)
cfg["DQAFullSupervisedBaseline"] = {
    "source": "04_dqa_ver2 8h schedule",
    "phase_split": False,
    "dqa": False,
    "ssod_target": False,
    "epochs": TOTAL_SUPERVISED_EPOCHS,
    "train_images": full_manifest["total_images"],
    "batch_size": BATCH_SIZE,
    "sample_equivalent_epochs_reference": sample_equivalent_epochs,
}
CONFIG_PATH = setup.write_config(CONFIG_PATH.name, cfg)
print(CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8")[:2500])

## 6. Dry Run Command

In [ ]:
def train_command() -> list[str]:
    if GPUS > 1:
        return [
            str(PYTHON_BIN),
            "-m",
            "torch.distributed.run",
            "--nproc_per_node",
            str(GPUS),
            "--master_port",
            str(MASTER_PORT),
            "train.py",
            "--cfg",
            str(CONFIG_PATH.resolve()),
        ]
    return [str(PYTHON_BIN), "train.py", "--cfg", str(CONFIG_PATH.resolve())]


cmd = train_command()
print(" ".join(cmd))
print("cwd:", ET_ROOT)
print("run output:", RUN_ROOT / RUN_NAME)
print("final checkpoint:", FINAL_CHECKPOINT)

## 7. Run Full Supervised Training

`RUN_FULL_SUPERVISED=True` のまま実行すると、foregroundで学習を開始します。CUDAが見えていない場合は既定では実行しません。

In [ ]:
IMPORTANT_OUTPUT = re.compile(
    r"(Epoch|Starting training|Transferred|Optimizer|Scanning|Results saved|Training complete|"
    r"Traceback|RuntimeError|Exception|Error|out of memory|CUDA error)",
    re.IGNORECASE,
)


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return None
    try:
        return int(text)
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


def compact_line(line: str, limit: int = 240) -> str:
    text = line.replace("\r", "").strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def tail_lines(path: Path, lines: int = 80) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True, check=True)
        return result.stdout.splitlines()
    except (FileNotFoundError, subprocess.CalledProcessError):
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


def checkpoint_present(path: Path) -> bool:
    return path.exists() and path.stat().st_size >= 1024 * 1024


def finalize_checkpoint() -> None:
    last_ckpt = RUN_ROOT / RUN_NAME / "weights" / "last.pt"
    if not checkpoint_present(last_ckpt):
        raise FileNotFoundError(f"Expected trained checkpoint was not found: {last_ckpt}")
    GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_ckpt, FINAL_CHECKPOINT)
    history = [
        {
            "kind": "full_supervised",
            "run_name": RUN_NAME,
            "epochs": TOTAL_SUPERVISED_EPOCHS,
            "train_images": full_manifest["total_images"],
            "checkpoint": str(FINAL_CHECKPOINT.resolve()),
            "protocol": "dqa_full_supervised_04_8h_v1",
            "completed_utc": datetime.now(timezone.utc).isoformat(),
        }
    ]
    (WORK_ROOT / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    print("Final checkpoint:", FINAL_CHECKPOINT)


free_gib = shutil.disk_usage(WORK_ROOT).free / 1024**3 if WORK_ROOT.exists() else shutil.disk_usage(DQA_ROOT).free / 1024**3
if free_gib < MIN_FREE_GIB:
    raise RuntimeError(f"Only {free_gib:.2f} GiB free; requested at least {MIN_FREE_GIB:.2f} GiB.")

if RUN_FULL_SUPERVISED and AVAILABLE_CUDA_GPUS < 1 and not ALLOW_CPU_TRAINING:
    print(
        "No CUDA GPU is visible, so the full supervised run was not started. "
        "Use a GPU runtime, or set ALLOW_CPU_TRAINING = True only for a tiny debug run."
    )
elif RUN_FULL_SUPERVISED:
    existing_pid = read_pid(PID_PATH)
    existing_state = pid_state(existing_pid)
    if existing_state not in {"missing", "zombie"}:
        raise RuntimeError(
            f"Full supervised runner is already active with PID {existing_pid} (state={existing_state})."
        )

    if FINAL_CHECKPOINT.exists() and checkpoint_present(FINAL_CHECKPOINT) and not FORCE_RETRAIN:
        print("Reusing completed full supervised checkpoint:", FINAL_CHECKPOINT)
    else:
        if not APPEND_TRAIN_LOG and TRAIN_LOG.exists():
            TRAIN_LOG.unlink()
        RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
        TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
        cmd = train_command()
        print("Running:", " ".join(cmd))
        print("Train log:", TRAIN_LOG)
        with RUNNER_LOG.open("a", encoding="utf-8") as runner_log, TRAIN_LOG.open("a", encoding="utf-8") as train_log:
            runner_log.write("\n" + "=" * 100 + "\n")
            runner_log.write(datetime.now(timezone.utc).isoformat() + "\n")
            runner_log.write(" ".join(cmd) + "\n")
            runner_log.flush()
            train_log.write("\n" + "=" * 100 + "\n")
            train_log.write(" ".join(cmd) + "\n")
            train_log.flush()

            process = subprocess.Popen(
                cmd,
                cwd=ET_ROOT,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            PID_PATH.write_text(str(process.pid) + "\n", encoding="utf-8")
            print("PID:", process.pid)
            last_status = time.monotonic()
            recent = deque(maxlen=120)
            assert process.stdout is not None
            try:
                for line in process.stdout:
                    train_log.write(line)
                    train_log.flush()
                    recent.append(line.rstrip("\n"))
                    if IMPORTANT_OUTPUT.search(line):
                        print(compact_line(line))
                    now = time.monotonic()
                    if now - last_status >= STATUS_EVERY_SECONDS:
                        print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), compact_line(line))
                        last_status = now
                return_code = process.wait()
            finally:
                if PID_PATH.exists():
                    PID_PATH.unlink()
            if return_code != 0:
                print("Last training-output lines:")
                for row in recent:
                    print(compact_line(row, limit=500))
                raise RuntimeError(f"Training failed with exit code {return_code}; see {TRAIN_LOG}")
        finalize_checkpoint()

if FINAL_CHECKPOINT.exists():
    print("Ready:", FINAL_CHECKPOINT)

## 8. Inspect Progress and Artifacts

In [ ]:
history = load_json(WORK_ROOT / "history.json", [])
status = {
    "pid": read_pid(PID_PATH),
    "pid_state": pid_state(read_pid(PID_PATH)),
    "workspace": str(WORK_ROOT),
    "run_dir": str(RUN_ROOT / RUN_NAME),
    "final_checkpoint_exists": FINAL_CHECKPOINT.exists(),
    "final_checkpoint_size_gib": round(FINAL_CHECKPOINT.stat().st_size / 1024**3, 3) if FINAL_CHECKPOINT.exists() else 0,
    "history_entries": len(history) if isinstance(history, list) else 0,
    "train_log": str(TRAIN_LOG),
    "runner_log": str(RUNNER_LOG),
    "free_gib": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2) if WORK_ROOT.exists() else None,
}
display(pd.DataFrame([status]))

print("Runner log tail:")
for line in tail_lines(RUNNER_LOG, 30):
    print(line)
print("\nTrain log tail:")
for line in tail_lines(TRAIN_LOG, 30):
    print(line)

## 9. Paper-Style Evaluation

既定はquick確認用に `cloudy` のみです。最終表を作るときは `EVAL_SPLITS = "cloudy,overcast,rainy,snowy,total"` に変更します。

In [ ]:
RUN_EVAL = False
EVAL_SPLITS = "cloudy"
EVAL_BATCH_SIZE = 32
EVAL_EXTRA_ARGS: list[str] = ["--no-plots"]

eval_checkpoints = []
if FINAL_CHECKPOINT.exists():
    eval_checkpoints.append(f"full_supervised_56epochs={FINAL_CHECKPOINT}")

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
]
for spec in eval_checkpoints:
    eval_cmd.extend(["--checkpoint", spec])
eval_cmd.extend(EVAL_EXTRA_ARGS)

print(" ".join(eval_cmd))
if RUN_EVAL and eval_checkpoints:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
elif RUN_EVAL:
    print("Skipping evaluation because no full supervised checkpoint exists yet.")

summary_path = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("No full-supervised paper-protocol summary yet:", summary_path)

## 10. Compare Against DQA/FedSTO Summaries

In [ ]:
comparison_paths = {
    "DQA-CWA corrected 8h": DQA_ROOT / "efficientteacher_dqa_cwa_corrected_12h" / "validation_reports" / "paper_protocol_eval_summary.csv",
    "DQA-CWA v2 8h": DQA_ROOT / "efficientteacher_dqa_ver2" / "validation_reports" / "paper_protocol_eval_summary.csv",
    "DQA-CWA v2 scene 12h": DQA_ROOT / "efficientteacher_dqa_ver2_scene_12h" / "validation_reports" / "paper_protocol_eval_summary.csv",
    "FedSTO": NAV_ROOT / "efficientteacher_fedsto" / "validation_reports" / "paper_protocol_eval_summary.csv",
    "Full supervised 00": WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv",
}

frames = []
for method, path in comparison_paths.items():
    if path.exists():
        df = pd.read_csv(path)
        df.insert(0, "method", method)
        frames.append(df)
    else:
        print("missing:", method, path)

if frames:
    display(pd.concat(frames, ignore_index=True))
else:
    print("No comparable paper-protocol summaries found yet.")

## 11. Artifact Index

In [ ]:
def artifact_row(path: Path, label: str) -> dict:
    exists = path.exists()
    return {
        "label": label,
        "path": str(path),
        "exists": exists,
        "modified_utc": modified_utc(path),
    }


artifact_rows = [
    artifact_row(FULL_TRAIN_LIST, "full_train_list"),
    artifact_row(FULL_MANIFEST_PATH, "full_manifest"),
    artifact_row(CONFIG_PATH, "train_config"),
    artifact_row(RUN_ROOT / RUN_NAME, "run_dir"),
    artifact_row(FINAL_CHECKPOINT, "final_checkpoint"),
    artifact_row(WORK_ROOT / "history.json", "history"),
    artifact_row(TRAIN_LOG, "train_log"),
    artifact_row(RUNNER_LOG, "runner_log"),
    artifact_row(WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv", "paper_eval_summary_csv"),
    artifact_row(DATA_ROOT, "full_supervised_data"),
]
display(pd.DataFrame(artifact_rows))